In [1]:
from langgraph.graph import StateGraph, START,END, add_messages
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage,HumanMessage
from langgraph.checkpoint.memory import MemorySaver

In [2]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="qwen2.5-coder:7b", base_url="http://localhost:11434")


c:\numerix-python-project\ai-code-assistance\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class JokeState(TypedDict):
  topic: str
  joke: str
  explanation: str
  

In [4]:
def generate_joke(state:JokeState):
  prompt = f"Tell me a joke about {state['topic']}"
  response = llm.invoke([HumanMessage(content=prompt)])
  return {"joke": response.content}

def explain_joke(state:JokeState):
  prompt = f"Explain this joke: {state['joke']}"
  response = llm.invoke([HumanMessage(content=prompt)])
  return {"explanation": response.content}

In [5]:
graph = StateGraph(JokeState)
graph.add_node("generate_joke", generate_joke)
graph.add_node("explain_joke", explain_joke)
graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "explain_joke")
graph.add_edge("explain_joke", END)
checkpointers = MemorySaver()
workflow = graph.compile(checkpointer=checkpointers)

In [6]:
config = {"configurable": {"thread_id": "1"}}
workflow.invoke({"topic": "Pizza"}, config=config)

{'topic': 'Pizza',
 'joke': "Sure! Here's a joke for you:\n\nWhy did the pizza cross the road?\n\nTo get to the other side... and then it got a cheese on the other side!",
 'explanation': 'The joke is a classic play on the traditional "Why did the chicken cross the road?" type of joke, with the punchline being unexpected because instead of just crossing the road, the pizza gets a cheese. It\'s a light-hearted pun that aims to make people smile.'}

In [11]:
list(workflow.get_state_history(config))

[StateSnapshot(values={'topic': 'Pizza', 'joke': "Sure! Here's a joke for you:\n\nWhy did the pizza cross the road?\n\nTo get to the other side... and then it got a cheese on the other side!", 'explanation': 'The joke is a classic play on the traditional "Why did the chicken cross the road?" type of joke, with the punchline being unexpected because instead of just crossing the road, the pizza gets a cheese. It\'s a light-hearted pun that aims to make people smile.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1461fd-fbbd-63ee-8002-3cb613906b6e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-05-02T12:10:13.675500+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1461fd-8e93-6a7f-8001-69f085930ee5'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'Pizza', 'joke': "Sure! Here's a joke for you:\n\nWhy did the pizza cross the road?\n\nTo get to the other

In [7]:
workflow.get_state(config)

StateSnapshot(values={'topic': 'Pizza', 'joke': "Sure! Here's a joke for you:\n\nWhy did the pizza cross the road?\n\nTo get to the other side... and then it got a cheese on the other side!", 'explanation': 'The joke is a classic play on the traditional "Why did the chicken cross the road?" type of joke, with the punchline being unexpected because instead of just crossing the road, the pizza gets a cheese. It\'s a light-hearted pun that aims to make people smile.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1461fd-fbbd-63ee-8002-3cb613906b6e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-05-02T12:10:13.675500+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1461fd-8e93-6a7f-8001-69f085930ee5'}}, tasks=(), interrupts=())

In [9]:
config_2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({"topic": "Pasta"}, config=config_2)
list(workflow.get_state_history(config_2))

[StateSnapshot(values={'topic': 'Pasta', 'joke': "Sure! Here's a light-hearted joke for you:\n\nWhy did the pasta cross the road?\n\nTo get to the other side... but it got stuck in the tomato sauce!", 'explanation': 'The joke is a play on the classic "Why did the chicken cross the road?" which is a common setup for a humoristic question. In this case, the punchline is unexpected and humorous because instead of the pasta simply crossing over to the other side, it faces an obstacle in the form of getting stuck in tomato sauce. The humor comes from the twist and the unexpected difficulty that the pasta encounters despite its simple intention.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f146218-7b44-6dca-8002-08957747bdc9'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-05-02T12:22:04.980150+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f146217-f8d5-665f-8001-82